In [4]:
import kagglehub
path = kagglehub.dataset_download("kazanova/sentiment140")

100%|██████████| 80.9M/80.9M [00:00<00:00, 196MB/s]

Extracting files...


In [5]:
import os
import pandas as pd

# List the contents of the new dataset directory
print(f"New dataset downloaded to: {path}")
print("Contents of the dataset directory:")
for root, dirs, files in os.walk(path):
    for name in files:
        print(os.path.join(root, name))
    for name in dirs:
        print(os.path.join(root, name))

New dataset downloaded to: /root/.cache/kagglehub/datasets/kazanova/sentiment140/versions/2
Contents of the dataset directory:
/root/.cache/kagglehub/datasets/kazanova/sentiment140/versions/2/training.1600000.processed.noemoticon.csv


The `sentiment140` dataset typically contains a CSV file named `training.1600000.processed.noemoticon.csv`. Let's load this file into a pandas DataFrame and inspect its structure.

In [6]:
# Define the path to the CSV file based on the typical 'sentiment140' structure
csv_file_path = os.path.join(path, 'training.1600000.processed.noemoticon.csv')

# The dataset has no header, so we define column names manually
column_names = ['target', 'id', 'date', 'flag', 'user', 'text']

try:
    df_sentiment = pd.read_csv(csv_file_path, encoding='ISO-8859-1', names=column_names)
    print(f"Successfully loaded {len(df_sentiment)} rows.")
    display(df_sentiment.head())
except FileNotFoundError:
    print(f"Error: '{csv_file_path}' not found. Please verify the filename and path.")
except Exception as e:
    print(f"An error occurred while loading the CSV data: {e}")

Successfully loaded 1600000 rows.


,target,id,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [7]:
# Display basic information about the DataFrame
print("\nDataFrame Info:")
df_sentiment.info()

print("\nMissing values:")
print(df_sentiment.isnull().sum())

print("\nTarget variable distribution:")
print(df_sentiment['target'].value_counts())


DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1600000 entries, 0 to 1599999
Data columns (total 6 columns):
 #   Column  Non-Null Count    Dtype 
---  ------  --------------    ----- 
 0   target  1600000 non-null  int64 
 1   id      1600000 non-null  int64 
 2   date    1600000 non-null  object
 3   flag    1600000 non-null  object
 4   user    1600000 non-null  object
 5   text    1600000 non-null  object
dtypes: int64(2), object(4)
memory usage: 73.2+ MB

Missing values:
target    0
id        0
date      0
flag      0
user      0
text      0
dtype: int64

Target variable distribution:
target
0    800000
4    800000
Name: count, dtype: int64


### Text Tokenization and Padding

To feed text data into a deep learning model, we need to convert the words into numerical representations. This process involves:
1.  **Tokenization**: Assigning a unique integer ID to each word.
2.  **Padding**: Ensuring all input sequences have the same length by adding zeros.

We will also split the dataset into training and testing sets.

In [11]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Parameters for tokenization and padding
VOCAB_SIZE = 50000  # Consider the top 50,000 most frequent words
MAX_SEQUENCE_LENGTH = 100 # Maximum length of a tweet

# Initialize tokenizer
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<unk>')
tokenizer.fit_on_texts(df_sentiment['cleaned_text'])

# Convert text to sequences of integers
sequences = tokenizer.texts_to_sequences(df_sentiment['cleaned_text'])

# Pad sequences to ensure uniform length
padded_sequences = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')

# Prepare labels
labels = df_sentiment['sentiment'].values

print(f"Shape of padded sequences: {padded_sequences.shape}")
print(f"Shape of labels: {labels.shape}")

# Display a sample of tokenized and padded sequences
print("\nSample of padded sequences (first 5):")
display(padded_sequences[:5])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"\nTraining set shape (X_train): {X_train.shape}")
print(f"Testing set shape (X_test): {X_test.shape}")
print(f"Training labels shape (y_train): {y_train.shape}")
print(f"Testing labels shape (y_test): {y_test.shape}")

/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Shape of padded sequences: (1600000, 100)
Shape of labels: (1600000,)

Sample of padded sequences (first 5):


array([[    5,   102,     5,  1207,     8,  3426,    49,   863,  9708,
           13,  1842,    32,     3,    41,    10,   384,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,
            0],
       [    9,   772,    19,   107,    47,   538,   178,   542,   120,
         2000,    10,     7,   282,   522,    78,     5,  2274,   145,
           43,   259,  1186,     0,     0,     0,     0,     


Training set shape (X_train): (1280000, 100)
Testing set shape (X_test): (320000, 100)
Training labels shape (y_train): (1280000,)
Testing labels shape (y_test): (320000,)


### Build the Deep Learning Model

We will construct a simple yet effective deep learning model for sentiment analysis using TensorFlow/Keras. The model will include:

1.  **Embedding Layer**: To convert word indices into dense, fixed-size vectors.
2.  **GlobalAveragePooling1D**: To downsample the input by taking the average over the time dimension, which is good for reducing dimensionality while retaining information.
3.  **Dense Layers**: Fully connected layers for classification.

Since this is a binary classification problem (positive or negative sentiment), the output layer will have a single neuron with a sigmoid activation function.

In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Model Parameters
EMBEDDING_DIM = 128 # Dimension of the dense embedding

# Define the model
model = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH),
    GlobalAveragePooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5), # Add dropout for regularization
    Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

# Display model summary
print("Model Summary:")
model.summary()

Model Summary:


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Train the Model

Now that the model is defined and compiled, let's train it using our preprocessed training data. We will also monitor its performance on the validation set (a portion of the training data) to detect overfitting.

In [14]:
# Define training parameters
BATCH_SIZE = 1024 # Larger batch size due to the large dataset
EPOCHS = 1 # Start with a few epochs, can be increased if needed

# Train the model
history = model.fit(
    X_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1, # Use 10% of training data for validation
    verbose=1
)

print("\nModel training complete.")

1125/1125 ━━━━━━━━━━━━━━━━━━━━ 114s 101ms/step - accuracy: 0.7632 - loss: 0.5031 - val_accuracy: 0.7678 - val_loss: 0.4814

Model training complete.


### Evaluate the Model

After training, it's crucial to evaluate the model's performance on the unseen test set to get an unbiased estimate of its generalization capability.

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test, batch_size=BATCH_SIZE, verbose=1)

print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Plot training history
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

### Save Model and Tokenizer

To use the trained model and tokenizer in a Streamlit app, we need to save them to disk. This ensures they can be loaded independently without retraining or re-fitting.

In [15]:
import pickle

# Save the Keras model
model.save('sentiment_model.h5')
print("Model saved as sentiment_model.h5")

# Save the Tokenizer
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
print("Tokenizer saved as tokenizer.pickle")

Model saved as sentiment_model.h5
Tokenizer saved as tokenizer.pickle


### Create Streamlit App

Now, let's create the Streamlit application. This app will:

1.  Load the saved model and tokenizer.
2.  Provide a text input field for the user.
3.  Preprocess the user's input text using the `clean_text` function and the loaded tokenizer.
4.  Make a sentiment prediction using the loaded model.
5.  Display the predicted sentiment.

Copy the code below into a file named `app.py` in your local environment.

In [22]:
%%writefile app.py

import streamlit as st
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle
import re
import numpy as np

# --- Page Configuration --- #
st.set_page_config(layout="wide", page_title="Tweet Sentiment Analyzer 🐦")

# --- Configuration --- #
MAX_SEQUENCE_LENGTH = 100 # Must match the length used during training

# --- Load Model and Tokenizer --- #
@st.cache_resource
def load_resources():
    # Load the model
    model = tf.keras.models.load_model('sentiment_model.h5')
    # Load the tokenizer
    with open('tokenizer.pickle', 'rb') as handle:
        tokenizer = pickle.load(handle)
    return model, tokenizer

model, tokenizer = load_resources()

# --- Text Cleaning Function (must match preprocessing) --- #
def clean_text(text):
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove user mentions (e.g., @username)
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags (keeping the text if desired, here we remove the #)
    text = re.sub(r'#', '', text)
    # Remove special characters and numbers, keeping only letters and spaces
    text = re.sub(r'[^A-Za-z\s]', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Convert to lowercase
    text = text.lower()
    return text

# --- Prediction Function --- #
def predict_sentiment(text):
    cleaned = clean_text(text)
    if not cleaned:
        return None # Handle cases where text becomes empty after cleaning
    sequence = tokenizer.texts_to_sequences([cleaned])
    padded_sequence = pad_sequences(sequence, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
    prediction = model.predict(padded_sequence, verbose=0)[0][0]
    return prediction

# --- Streamlit UI --- #
st.title('🐦 Tweet Sentiment Analyzer')

# Sidebar
st.sidebar.header('About')
st.sidebar.info(
    "This app uses a Deep Learning model to predict the sentiment of tweets (positive or negative). "
    "The model was trained on a large dataset of tweets."
)
st.sidebar.header('Instructions')
st.sidebar.markdown(
    "1. Enter a tweet into the text area.\n"
    "2. Click 'Analyze Sentiment' to see the prediction.\n"
    "3. The sentiment score ranges from 0 (negative) to 1 (positive)."
)

st.write('Enter a tweet below to predict its sentiment.')

user_input = st.text_area('✍️ Enter your tweet here:', '', height=150)

if st.button('✨ Analyze Sentiment'):
    if user_input:
        with st.spinner('Analyzing sentiment...'):
            sentiment_score = predict_sentiment(user_input)

            if sentiment_score is not None:
                st.subheader('Analysis Result:')
                if sentiment_score >= 0.6:
                    st.balloons()
                    st.success(f'🎉 Positive Sentiment! (Score: {sentiment_score:.2f})')
                elif sentiment_score <= 0.4:
                    st.error(f'😔 Negative Sentiment! (Score: {sentiment_score:.2f})')
                else:
                    st.info(f'😐 Neutral/Mixed Sentiment (Score: {sentiment_score:.2f})')
            else:
                st.warning('The entered text was too short or contained no meaningful words after cleaning.')
    else:
        st.warning('Please enter some text to analyze.')


Overwriting app.py


### Instructions to Run the Streamlit App

1.  **Download the files**: After executing the cell above, two files will be created in your Colab environment: `sentiment_model.h5` and `tokenizer.pickle`, and a file named `app.py`.
    You'll need to download these files to your local machine where you want to run the Streamlit app. You can do this by clicking the folder icon on the left panel, navigating to these files, right-clicking, and selecting 'Download'.

2.  **Install Streamlit**: If you haven't already, install Streamlit in your local Python environment:
    ```bash
    pip install streamlit
    ```

3.  **Run the App**: Open your terminal or command prompt, navigate to the directory where you saved `app.py`, `sentiment_model.h5`, and `tokenizer.pickle`, and then run the following command:
    ```bash
    streamlit run app.py
    ```

4.  **Access the App**: Streamlit will open a new tab in your web browser with the sentiment analysis application. You can now type in text and get sentiment predictions!

In [17]:
!pip install --ignore-installed blinker
!pip install streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 135.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 152.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 7.9 MB/s eta 0:00:00


In [18]:
!pip install streamlit pyngrok

In [19]:
from pyngrok import ngrok
ngrok.set_auth_token("3H2nxZtP4iC5L9tX9K97OPLut9W_4JsZrRVF5aRFQpQCCePy1")

In [23]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://vanity-amperage-commodore.ngrok-free.dev" -> "http://localhost:8501"


In [24]:
# Run the Streamlit app
# This will provide a public URL to access the app
!streamlit run app.py &




2026-08-25 12:59:32.759 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.66.144.110:8501

I0000 00:00:1787662776.078259    5168 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
I0000 00:00:1787662777.473538    5168 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1787662778.669275    5168 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UN

In [10]:
# Install TensorFlow
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.9/572.9 MB 810.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 67.5 MB/s eta 0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0


### Data Preprocessing

For sentiment analysis, we need to clean the text data by removing noise such as URLs, mentions, hashtags, and special characters. We will also convert the target labels from `0` and `4` to `0` and `1` respectively, where `0` represents negative sentiment and `1` represents positive sentiment.

In [8]:
import re

def clean_text(text):
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove user mentions (e.g., @username)
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags (keeping the text if desired, here we remove the #)
    text = re.sub(r'#', '', text)
    # Remove special characters and numbers, keeping only letters and spaces
    text = re.sub(r'[^A-Za-z\s]', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Convert to lowercase
    text = text.lower()
    return text

# Apply the cleaning function to the 'text' column
df_sentiment['cleaned_text'] = df_sentiment['text'].apply(clean_text)

# Convert target labels: 0 -> 0 (negative), 4 -> 1 (positive)
df_sentiment['sentiment'] = df_sentiment['target'].replace({0: 0, 4: 1})

print("Original text samples:")
display(df_sentiment[['text', 'cleaned_text', 'sentiment']].head())

print("\nUpdated target variable distribution:")
print(df_sentiment['sentiment'].value_counts())

Original text samples:


,text,cleaned_text,sentiment
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",a thats a bummer you shoulda got david carr of...,0
1,is upset that he can't update his Facebook by ...,is upset that he cant update his facebook by t...,0
2,@Kenichan I dived many times for the ball. Man...,i dived many times for the ball managed to sav...,0
3,my whole body feels itchy and like its on fire,my whole body feels itchy and like its on fire,0
4,"@nationwideclass no, it's not behaving at all....",no its not behaving at all im mad why am i her...,0



Updated target variable distribution:
sentiment
0    800000
1    800000
Name: count, dtype: int64
